In [1]:
!pip install -U transformers datasets huggingface_hub tqdm pyarrow psutil
from transformers import AutoTokenizer
import datasets
from datasets import load_dataset
import torch
from tqdm import tqdm
import huggingface_hub
import pyarrow
import multiprocessing
import math
from itertools import chain, islice
import json
import os
from huggingface_hub import hf_hub_download
from huggingface_hub import HfApi, HfFileSystem
from huggingface_hub.utils import RepositoryNotFoundError
from sklearn.preprocessing import RobustScaler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 68.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 29.0 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 67.5 MB/s eta 0:00:00:00:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingfac

In [ ]:
import numpy as np
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token = hf_token)

file_path = "casey-martin/CommonLit-Ease-of-Readability"
dataset = load_dataset(path = file_path, split = "train+validation+test", token = hf_token)
train_dataset = load_dataset(path = file_path, split = "train+validation", token = hf_token)
validation_dataset = load_dataset(path = file_path, split = "test", token = hf_token)

BT_scores = dataset["BT_easiness"]
lower_bound = np.percentile(BT_scores, 1)
upper_bound = np.percentile(BT_scores, 99)
score_range = upper_bound - lower_bound

def cap_and_normalize(example):
    val = np.array(example["BT_easiness"])
    
    # Clip values to bounds (Capping)
    clipped_val = max(lower_bound, min(val, upper_bound))
    
    # Min-max scale the clipped value
    example["BT_easiness_normalized"] = (clipped_val - lower_bound) / score_range
    return example

train_dataset = train_dataset.map(cap_and_normalize)
validation_dataset = validation_dataset.map(cap_and_normalize)

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base", use_fast = True)

def tokenizing_function(example):
    input = tokenizer(example["Excerpt"], 
        padding = "max_length", 
        max_length = 512, 
        truncation = True, 
        return_tensors = "pt", 
        return_attention_mask = True, 
        return_token_type_ids = False
    )
    input["input_ids"] = input["input_ids"].squeeze()
    input["attention_mask"] = input["attention_mask"].squeeze()
    input["labels"] = example["BT_easiness_normalized"]
    return input

train_dataset = train_dataset.map(tokenizing_function, remove_columns = ['ID', 'Author', 'Title', 'Anthology', 'URL', 'Pub Year', 'Categ', 'Sub Cat', 'Lexile Band', 'Location', 'License', 'MPAA Max', 'MPAA #Max', 'MPAA# Avg', 'Excerpt', 'Google WC', 'Sentence Count', 'Paragraphs', 'BT_easiness', 's.e.', 'Flesch-Reading-Ease', 'Flesch-Kincaid-Grade-Level', 'Automated Readability Index', 'SMOG Readability', 'New Dale-Chall Readability Formula', 'CAREC', 'CAREC_M', 'CML2RI', '__index_level_0__', 'BT_easiness_normalized']) 
validation_dataset = validation_dataset.map(tokenizing_function, remove_columns = ['ID', 'Author', 'Title', 'Anthology', 'URL', 'Pub Year', 'Categ', 'Sub Cat', 'Lexile Band', 'Location', 'License', 'MPAA Max', 'MPAA #Max', 'MPAA# Avg', 'Excerpt', 'Google WC', 'Sentence Count', 'Paragraphs', 'BT_easiness', 's.e.', 'Flesch-Reading-Ease', 'Flesch-Kincaid-Grade-Level', 'Automated Readability Index', 'SMOG Readability', 'New Dale-Chall Readability Formula', 'CAREC', 'CAREC_M', 'CML2RI', '__index_level_0__', 'BT_easiness_normalized']) 

Map:   0%|          | 0/4251 [00:00<?, ? examples/s]

Map:   0%|          | 0/473 [00:00<?, ? examples/s]

In [23]:

api.create_repo(repo_id = "JamesResearch1216/BT_Easiness_Data", repo_type = "dataset", exist_ok = True)

train_parquet = train_dataset.to_parquet("train.parquet")
validation_parquet = validation_dataset.to_parquet("validation.parquet")

api.upload_file(
    path_or_fileobj = "train.parquet",
    path_in_repo = "data/train.parquet",
    repo_id = "JamesResearch1216/BT_Easiness_Data",
    repo_type = "dataset"
)

api.upload_file(
    path_or_fileobj = "validation.parquet",
    path_in_repo = "data/validation.parquet",
    repo_id = "JamesResearch1216/BT_Easiness_Data",
    repo_type = "dataset"
)


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/JamesResearch1216/BT_Easiness_Data/commit/701b9a4646107a94b96e24b4828ccffe5110ee89', commit_message='Upload data/validation.parquet with huggingface_hub', commit_description='', oid='701b9a4646107a94b96e24b4828ccffe5110ee89', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/JamesResearch1216/BT_Easiness_Data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='JamesResearch1216/BT_Easiness_Data'), pr_revision=None, pr_num=None)